In [ ]:
# 0.1 Environment + packages
import importlib.util, os, subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
for mod, pkg in [("lxml", "lxml"), ("networkx", "networkx"), ("tqdm", "tqdm"), ("psutil", "psutil"),
                 ("torch", "torch"), ("torch_geometric", "torch_geometric"), ("pyarrow", "pyarrow")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)


In [ ]:
# 0.2 Safe Drive mount (never deletes a mounted Drive)
if IN_COLAB:
    import shutil
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        if os.path.exists("/content/drive"):
            shutil.rmtree("/content/drive")
        drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
# 0.3 Configuration
import glob, gzip, json, time, shutil, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)

BASE_ROOT = os.environ.get("FCA_ROOT", "/content/drive/MyDrive/FCA_PAD_v2" if IN_COLAB
                           else os.path.abspath("FCA_PAD_v2"))
LOGS = {"BPIC20_D": "DomesticDeclarations.xes", "BPIC20_I": "BPIC20_InternationalDeclarations.xes",
        "BPIC20_PR": "BPIC20_PrepaidTravelCost.xes", "BPIC20_R": "BPIC20_RequestForPayment.xes"}
LOGS_TO_RUN = os.environ.get("GW_LOGS", ",".join(LOGS)).split(",")
SUMMARY_DIR = f"{BASE_ROOT}/16_gama_vs_fcapad"
os.makedirs(SUMMARY_DIR, exist_ok=True)
REPO_DIR = os.environ.get("BPAD_DIR", "/content/BPAD" if IN_COLAB else os.path.abspath("BPAD"))
REPO_URL = "https://github.com/guanwei49/BPAD.git"

RUN_GAMA = True
RUN_WAKE = False          # weakly supervised (10% labels); ~3-4x slower than GAMA
GAMA_EPOCHS = int(os.environ.get("GW_EPOCHS", 20))   # benchmark default: 20
WAKE_LABEL_PERCENT = 0.1  # benchmark default
SEED = 0
BOOTSTRAP_B = 200

CASE_COL, ACTIVITY_COL, TIME_COL = "case:concept:name", "concept:name", "time:timestamp"
ROLE_COL, RESOURCE_COL = "org:role", "org:resource"
def log_paths(log):
    root = BASE_ROOT if log == "BPIC20_D" else f"{BASE_ROOT}/{log}"
    return root, {"events_injected": f"{root}/02_processed/events_injected.parquet",
                  "ground_truth": f"{root}/00_frozen/ground_truth_sealed.csv",
                  "split": f"{root}/00_frozen/case_split.csv",
                  "injected_scores": f"{root}/03_features/injected_scores.parquet",
                  "headline": f"{root}/05_results/07_headline_test.csv"}

AVAILABLE = []
for log in LOGS_TO_RUN:
    root, need = log_paths(log)
    ok = all(os.path.exists(p) for p in need.values())
    print(f"{log:10s} {'ready' if ok else 'SKIPPED - run the main notebook with LOG_NAME = ' + repr(log)}")
    if ok:
        AVAILABLE.append(log)
assert AVAILABLE, "No log is ready: run the main notebook first."


BPIC20_D   ready
BPIC20_I   ready
BPIC20_PR  ready
BPIC20_R   ready


## 1 · Benchmark code


In [ ]:
# 1.1 Clone the benchmark repo; patch the one numpy-2 incompatibility (.tostring -> .tobytes)
if not os.path.exists(f"{REPO_DIR}/baseline/GAMA/gama.py"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
for py in glob.glob(f"{REPO_DIR}/**/*.py", recursive=True):
    s = open(py, encoding="utf-8", errors="ignore").read()
    if ".tostring()" in s:
        open(py, "w", encoding="utf-8").write(s.replace(".tostring()", ".tobytes()"))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


torch 2.11.0+cu130 | CUDA: True


## 2 · Helpers: export to the benchmark format, train, evaluate with FCA-PAD metrics


In [ ]:
import gc, psutil
from utils.fs import EVENTLOG_DIR
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

def ram(tag=""):
    vm = psutil.virtual_memory()
    print(f"  [RAM {tag}] used {vm.used / 2**30:.1f} GB of {vm.total / 2**30:.1f} GB")

# Log header for the benchmark format. Written directly instead of parsing the whole XES
# (the export only carries activity, org:resource, org:role, timestamp).
def log_header(log):
    return {"extensions": [{"name": "Organizational", "prefix": "org", "uri": "http://www.xes-standard.org/org.xesext"},
                           {"name": "Time", "prefix": "time", "uri": "http://www.xes-standard.org/time.xesext"},
                           {"name": "Concept", "prefix": "concept", "uri": "http://www.xes-standard.org/concept.xesext"}],
            "global_attributes": {"event": {"time:timestamp": {"type": "date", "value": "1970-01-01T00:00:00+01:00"},
                                            "org:resource": {"type": "string", "value": "UNKNOWN"},
                                            "concept:name": {"type": "string", "value": "UNKNOWN"},
                                            "org:role": {"type": "string", "value": "UNKNOWN"}}},
            "classifiers": [{"name": "Activity classifier", "keys": ["concept:name"]}],
            "concept:name": {"type": "string", "value": log}}

# GAMA / WAKE are trained in a separate Python process per log: when it ends, all of its
# memory (dataset graphs, model, CUDA context) is returned, so RAM cannot build up across logs.
WORKER = f"{REPO_DIR}/_fcapad_worker.py"
open(WORKER, "w").write(r'''
import sys, time, numpy as np, pandas as pd, torch
repo, method, ds_name, cache, cases_csv, epochs, seed, lp = sys.argv[1:9]
sys.path.insert(0, repo)
from utils.dataset import Dataset
from utils.eval import cal_best_PRF
np.random.seed(int(seed)); torch.manual_seed(int(seed))
if method == "GAMA":
    from baseline.GAMA.gama import GAMA
    ds, ad = Dataset(ds_name, beta=0.005), GAMA(n_epochs=int(epochs))
else:
    from baseline.WAKE.wake import WAKE
    ds, ad = Dataset(ds_name, label_percent=float(lp)), WAKE()
t0 = time.time()
ad.fit(ds)
tr, _, _ = ad.detect(ds)
_, _, F, AP = cal_best_PRF(ds.case_target, tr)
print(f"  {method}: whole-log F={F:.3f} AP={AP:.3f} ({time.time() - t0:.0f}s)", flush=True)
ids = pd.read_csv(cases_csv, dtype=str)["case"].values
pd.DataFrame({"case": ids, "score": np.asarray(tr, float)}).to_csv(cache, index=False)
if method == "WAKE":
    lab = np.zeros(len(ids), bool); lab[np.asarray(ds.labeled_indices, int)] = True
    pd.DataFrame({"case": ids, "labelled": lab}).to_csv(cache.replace("_scores.csv", "_labelled.csv"), index=False)
''')

def load_world(log):
    root, need = log_paths(log)
    ev = pd.read_parquet(need["events_injected"])
    ev[TIME_COL] = pd.to_datetime(ev[TIME_COL], utc=True)
    ev[CASE_COL] = ev[CASE_COL].astype(str)
    gt = pd.read_csv(need["ground_truth"], dtype={CASE_COL: str})
    split = pd.read_csv(need["split"], dtype={CASE_COL: str}).set_index(CASE_COL)["split"]
    pxi = pd.read_parquet(need["injected_scores"], columns=[CASE_COL, "event_idx", "F"])
    pxi[CASE_COL] = pxi[CASE_COL].astype(str)
    last_F = pxi.loc[pxi.groupby(CASE_COL)["event_idx"].idxmax()].set_index(CASE_COL)["F"]
    cases = pd.DataFrame({CASE_COL: sorted(ev[CASE_COL].unique())})
    cases["split"] = cases[CASE_COL].map(split)
    g = gt.set_index(CASE_COL)
    cases["inj_type"] = cases[CASE_COL].map(g["type"]).fillna("-")
    cases["is_injected"] = (cases["inj_type"] != "-").astype(int)
    cases["gt_money"] = cases[CASE_COL].map(g["gt_money"]).fillna(0.0)
    cases["gt_loss"] = cases[CASE_COL].map(g["loss"]).fillna(0.0)
    cases["F"] = cases[CASE_COL].map(last_F).fillna(0.0)
    return root, need, ev, g, cases

def export_world(log, need, ev, g, cases):
    """A1/A2/A4 (inserted duplicate events) -> benchmark 'Insert'; A5 (approvals removed) -> 'SkipSequence'."""
    name = f"FCAPADinj{log.replace('_', '')}-0.05-1"
    log_attrs = log_header(log)
    cases_json = []
    for cid, grp in ev.sort_values([CASE_COL, "event_idx"]).groupby(CASE_COL, sort=True):
        evs = []
        for _, e in grp.iterrows():
            ts = e[TIME_COL].isoformat() if pd.notna(e[TIME_COL]) else None
            evs.append({"name": e[ACTIVITY_COL], "timestamp": ts, "timestamp_end": None,
                        "attributes": {"concept:name": e[ACTIVITY_COL], "org:resource": str(e[RESOURCE_COL]),
                                       "org:role": str(e[ROLE_COL]), "time:timestamp": ts}})
        if cid in g.index:
            r = g.loc[cid]
            if r["type"] == "A5":
                label = {"anomaly": "SkipSequence", "attr": {"size": 1, "start": int(r["anomaly_pos"]) - 1, "skipped": []}}
            else:
                idx = [int(r["anomaly_pos"]) - 1]
                if pd.notna(r["loss_pos"]) and int(r["loss_pos"]) != int(r["anomaly_pos"]):
                    idx.append(int(r["loss_pos"]) - 1)
                label = {"anomaly": "Insert", "attr": {"indices": sorted(idx)}}
        else:
            label = "normal"
        cases_json.append({"id": cid, "attributes": {"concept:name": cid, "label": label}, "events": evs})
    assert [c["id"] for c in cases_json] == cases[CASE_COL].tolist(), "case order mismatch"
    os.makedirs(f"{EVENTLOG_DIR}/cache", exist_ok=True)
    for f in (f"{EVENTLOG_DIR}/cache/{name}.pkl.gz", f"{EVENTLOG_DIR}/{name}.json.gz"):
        if os.path.exists(f):
            os.remove(f)
    with gzip.open(f"{EVENTLOG_DIR}/{name}.json.gz", "wt") as fh:
        json.dump({"attributes": log_attrs, "cases": cases_json}, fh)
    del cases_json; gc.collect()
    return name

def run_benchmark_method(method, ds_name, cases, out_dir):
    cache = f"{out_dir}/{method}_scores.csv"
    if not os.path.exists(cache):
        ids_csv = f"{out_dir}/_case_order.csv"
        pd.DataFrame({"case": cases[CASE_COL].values}).to_csv(ids_csv, index=False)
        tmp = f"{out_dir}/_{method}_scores_tmp.csv"
        cmd = [sys.executable, "-u", WORKER, REPO_DIR, method, ds_name, tmp, ids_csv,
               str(GAMA_EPOCHS), str(SEED), str(WAKE_LABEL_PERCENT)]
        print(f"  training {method} in a separate process ...", flush=True)
        p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        last = ""
        for line in p.stdout:                         # show progress without flooding the cell
            line = line.rstrip()
            if line and ("%|" not in line or "100%|" in line):
                print("   ", line[-160:], flush=True)
            last = line or last
        if p.wait() != 0:
            raise RuntimeError(f"{method} worker failed (exit {p.returncode}). Last output: {last[-300:]}\n"
                               "Exit -9 means the runtime ran out of RAM: use a High-RAM runtime or run one log "
                               "at a time with GW_LOGS.")
        os.replace(tmp, cache)                        # cache appears only after a complete run
    else:
        print(f"  {method}: loaded cached scores")
    s = pd.read_csv(cache, dtype={"case": str}).set_index("case")["score"]
    assert s.index.tolist() == cases[CASE_COL].tolist(), "cached scores do not match this log's cases"
    return s

def midpoint_rank(values, ref):
    ref = np.sort(np.asarray(ref)); lo = np.searchsorted(ref, values, "left")
    hi = np.searchsorted(ref, values, "right"); return np.clip((lo + hi) / 2 / len(ref), 0, 1)

def oracle_f1(y, s):
    p, r, _ = precision_recall_curve(y, s)
    f = 2 * p * r / np.clip(p + r, 1e-12, None)
    return float(np.nanmax(f[:-1]))

def value_at_k(df, s, col, k):
    v = df[col].values[np.argsort(-np.asarray(s), kind="stable")]
    return float(v[:k].sum() / max(v.sum(), 1e-9))

def evaluate(df, s):
    y = df["is_injected"].values
    out = {"PR_AUC": average_precision_score(y, s), "ROC_AUC": roc_auc_score(y, s),
           "F1_oracle": oracle_f1(y, s)}
    for k in (10, 20, 50, 100):
        out[f"V@{k}"] = value_at_k(df, s, "gt_money", k)
    out["LossRecall@50"] = value_at_k(df, s, "gt_loss", 50)
    return out

def boot_objective(df, s, B=BOOTSTRAP_B, seed=42):     # same selection objective as the main notebook
    rng = np.random.default_rng(seed); y, m, s = df.is_injected.values, df.gt_money.values, np.asarray(s)
    vals = []
    for _ in range(B):
        i = rng.integers(0, len(df), len(df))
        if y[i].sum() == 0:
            continue
        mm = m[i][np.argsort(-s[i], kind="stable")]
        vals.append((average_precision_score(y[i], s[i]) + mm[:50].sum() / max(mm.sum(), 1e-9)) / 2)
    return float(np.mean(vals))

def fuse(d, form, w):
    if form == "add":
        return w * d["A"].values + (1 - w) * d["F"].values
    if form == "mul":
        return d["A"].values * (w + (1 - w) * d["F"].values)
    if form == "noisy_or":
        return 1 - (1 - d["A"].values) * (1 - d["F"].values)
    return np.maximum(d["A"].values, d["F"].values)

GRID = np.round(np.arange(0, 1.0001, 0.05), 2)
CANDS = [("add", w) for w in GRID] + [("mul", w) for w in GRID] + [("noisy_or", None), ("max", None)]
FORM_RANK = {"add": 0, "mul": 1, "noisy_or": 2, "max": 3}


## 3 · Run every available log


In [ ]:
ALL_ROWS, ALL_TYPES = [], []
for log in AVAILABLE:
    print(f"\n========== {log} ==========")
    ram("start")
    root, need, ev, g, cases = load_world(log)
    out_dir = f"{root}/14_gama_wake"
    os.makedirs(out_dir, exist_ok=True)
    val_c = cases[cases.split == "val"].reset_index(drop=True)
    test_c = cases[cases.split == "test"].reset_index(drop=True)
    print(f"  test: {len(test_c)} cases, {int(test_c.is_injected.sum())} injected")
    methods = (["GAMA"] if RUN_GAMA else []) + (["WAKE"] if RUN_WAKE else [])
    need_export = any(not os.path.exists(f"{out_dir}/{m}_scores.csv") for m in methods)
    ds_name = export_world(log, need, ev, g, cases) if need_export else None
    del ev; gc.collect()                              # events are not needed after the export
    rows = []
    for method in methods:
        s = run_benchmark_method(method, ds_name, cases, out_dir)
        ref = val_c[CASE_COL].map(s).values
        for d in (val_c, test_c):
            d["A"] = midpoint_rank(d[CASE_COL].map(s).values, ref)
        rows.append({"system": f"{method} alone", **evaluate(test_c, test_c["A"].values)})
        cand = pd.DataFrame([{"form": f, "w": w, "obj": round(boot_objective(val_c, fuse(val_c, f, w)), 3)}
                             for f, w in CANDS])
        cand["rank"] = cand["form"].map(FORM_RANK)
        cand["centre"] = (cand["w"].fillna(0.5) - 0.5).abs()
        b = cand.sort_values(["obj", "rank", "centre"], ascending=[False, True, True]).iloc[0]
        w = None if pd.isna(b["w"]) else float(b["w"])
        fused = fuse(test_c, b["form"], w)
        rows.append({"system": f"{method} + FCA-PAD financial layer", "fusion": b["form"] + ("" if w is None else f" w={w}"),
                     **evaluate(test_c, fused)})
        neg = test_c[test_c.is_injected == 0]
        for t in ["A1", "A2", "A4", "A5"]:
            sub = pd.concat([neg, test_c[test_c.inj_type == t]])
            ALL_TYPES.append({"log": log, "type": t, "system": f"{method} alone",
                              "PR_AUC": average_precision_score(sub.is_injected, test_c.loc[sub.index, "A"])})
            ALL_TYPES.append({"log": log, "type": t, "system": f"{method} + financial layer",
                              "PR_AUC": average_precision_score(sub.is_injected, fused[sub.index])})
    ours = pd.read_csv(need["headline"])
    keep = {"v2 anomaly lane (next-event)": "FCA-PAD v2 anomaly lane", "v2 FCA-PAD": "FCA-PAD v2 (full)"}
    ours = ours[ours.system.isin(keep)].copy()
    ours["system"] = ours["system"].map(keep)
    ptype_path = f"{root}/05_results/07_per_type_test.csv"
    if os.path.exists(ptype_path):
        pt = pd.read_csv(ptype_path)
        lane_col = [c for c in pt.columns if c.startswith("A_NE_")]
        for _, r in pt.iterrows():
            if lane_col:
                ALL_TYPES.append({"log": log, "type": r["type"], "system": "FCA-PAD v2 anomaly lane", "PR_AUC": r[lane_col[0]]})
            ALL_TYPES.append({"log": log, "type": r["type"], "system": "FCA-PAD v2 (full)", "PR_AUC": r["R_new"]})
    res = pd.concat([ours, pd.DataFrame(rows)], ignore_index=True)
    res.insert(0, "log", log)
    res.round(4).to_csv(f"{out_dir}/comparison_test.csv", index=False)
    print(res.drop(columns="log").round(3).to_string(index=False))
    ALL_ROWS.append(res)
    del g, cases, val_c, test_c; gc.collect()
    if ds_name:                                       # remove the exported copy and its cache
        for f in glob.glob(f"{EVENTLOG_DIR}/{ds_name}.json.gz") + glob.glob(f"{EVENTLOG_DIR}/cache/{ds_name}*"):
            os.remove(f)
    ram("end")



========== BPIC20_D ==========
  [RAM start] used 1.4 GB of 12.7 GB
  test: 2100 cases, 105 injected
  training GAMA in a separate process ...
    **********training**********
    100%|██████████| 164/164 [00:50<00:00,  3.26it/s]
    [Epoch  1/20] [loss: 0.432625]
    100%|██████████| 164/164 [00:47<00:00,  3.48it/s]
    [Epoch  2/20] [loss: 0.116635]
    100%|██████████| 164/164 [00:48<00:00,  3.41it/s]
    [Epoch  3/20] [loss: 0.102070]
    100%|██████████| 164/164 [00:47<00:00,  3.45it/s]
    [Epoch  4/20] [loss: 0.094513]
    100%|██████████| 164/164 [00:48<00:00,  3.37it/s]
    [Epoch  5/20] [loss: 0.087704]
    100%|██████████| 164/164 [00:47<00:00,  3.46it/s]
    [Epoch  6/20] [loss: 0.078727]
    100%|██████████| 164/164 [00:48<00:00,  3.39it/s]
    [Epoch  7/20] [loss: 0.074576]
    100%|██████████| 164/164 [00:57<00:00,  2.85it/s]
    [Epoch  8/20] [loss: 0.072496]
    100%|██████████| 164/164 [00:48<00:00,  3.37it/s]
    [Epoch  9/20] [loss: 0.071580]
    100%|██████████| 1

## 4 · Across all logs


In [ ]:
allres = pd.concat(ALL_ROWS, ignore_index=True)
allres.round(4).to_csv(f"{SUMMARY_DIR}/all_logs_comparison.csv", index=False)
for metric in ["PR_AUC", "F1_oracle", "V@50", "LossRecall@50"]:
    piv = allres.pivot_table(index="system", columns="log", values=metric).round(3)
    piv["mean"] = piv.mean(axis=1).round(3)
    print(f"\n{metric} (TEST, ground-truth money):")
    print(piv.sort_values("mean", ascending=False).to_string())
    piv.to_csv(f"{SUMMARY_DIR}/{metric.replace('@', 'at')}_by_log.csv")

wins = []
for metric in ["PR_AUC", "F1_oracle", "V@50", "LossRecall@50"]:
    for log, gdf in allres.groupby("log"):
        best = gdf.loc[gdf[metric].idxmax()]
        a = gdf.set_index("system")[metric]
        wins.append({"log": log, "metric": metric, "best_system": best["system"],
                     "FCA-PAD_v2_full": a.get("FCA-PAD v2 (full)"), "GAMA_alone": a.get("GAMA alone"),
                     "FCA-PAD_beats_GAMA": bool(a.get("FCA-PAD v2 (full)", np.nan) > a.get("GAMA alone", np.inf))})
wins = pd.DataFrame(wins)
print("\nFCA-PAD v2 (full) vs GAMA alone, per log and metric:")
print(wins.round(3).to_string(index=False))
wins.to_csv(f"{SUMMARY_DIR}/fcapad_vs_gama_wins.csv", index=False)
n_cmp = wins["GAMA_alone"].notna().sum()
print(f"\nFCA-PAD v2 beats GAMA in {int(wins['FCA-PAD_beats_GAMA'].sum())}/{n_cmp} (log, metric) comparisons.")

if ALL_TYPES:
    types = pd.DataFrame(ALL_TYPES).pivot_table(index=["log", "type"], columns="system", values="PR_AUC").round(3)
    print("\nPer-type PR-AUC (TEST):")
    print(types.to_string())
    types.to_csv(f"{SUMMARY_DIR}/per_type_by_log.csv")



PR_AUC (TEST, ground-truth money):
log                             BPIC20_D  BPIC20_I  BPIC20_PR  BPIC20_R   mean
system                                                                        
FCA-PAD v2 (full)                  0.974     0.986      0.929     0.967  0.964
FCA-PAD v2 anomaly lane            0.972     0.986      0.897     0.972  0.957
GAMA + FCA-PAD financial layer     0.911     0.800      0.739     0.849  0.825
GAMA alone                         0.719     0.321      0.274     0.784  0.524

F1_oracle (TEST, ground-truth money):
log                             BPIC20_D  BPIC20_I  BPIC20_PR  BPIC20_R   mean
system                                                                        
FCA-PAD v2 anomaly lane            0.986     0.955      0.954     0.986  0.970
FCA-PAD v2 (full)                  0.954     0.955      0.865     0.972  0.936
GAMA + FCA-PAD financial layer     0.875     0.847      0.842     0.839  0.851
GAMA alone                         0.890     0.508      

In [ ]:
note = ("GAMA (and WAKE) were run with the benchmark's own code and default hyper-parameters, "
        "trained under the benchmark protocol (whole log, including injected val/test cases); "
        "WAKE additionally receives labels for 10% of anomalies. FCA-PAD was trained on clean "
        "training cases only. All systems are scored on the same frozen test set of each log with "
        "the same metrics. Fusion with the FCA-PAD financial layer is selected on validation "
        "(bootstrap mean of PR-AUC and V@50), never on test. Only BPIC20 logs with a single case "
        "amount and a request -> payment flow (D, I, PR, R) support the FCA-PAD metrics.")
open(f"{SUMMARY_DIR}/protocol_note.txt", "w").write(note)
print(note)
print("\nDONE. Cross-log results in", SUMMARY_DIR)


GAMA (and WAKE) were run with the benchmark's own code and default hyper-parameters, trained under the benchmark protocol (whole log, including injected val/test cases); WAKE additionally receives labels for 10% of anomalies. FCA-PAD was trained on clean training cases only. All systems are scored on the same frozen test set of each log with the same metrics. Fusion with the FCA-PAD financial layer is selected on validation (bootstrap mean of PR-AUC and V@50), never on test. Only BPIC20 logs with a single case amount and a request -> payment flow (D, I, PR, R) support the FCA-PAD metrics.

DONE. Cross-log results in /content/drive/MyDrive/FCA_PAD_v2/16_gama_vs_fcapad
